# 02. Filtering and Operators

## Quick notes
### What this covers
Comparison and logic operators, NULL and three-valued logic, LIKE/GLOB, IN, BETWEEN, CASE, COALESCE/NULLIF/IIF, operator precedence, and the number/text quirks of SQLite.
### Key points
- SQL has three truth values: TRUE, FALSE, UNKNOWN (NULL). `WHERE` keeps only TRUE.
- Test for NULL with `IS NULL` / `IS NOT NULL`, never `= NULL`.
- `IS` and `IS NOT` compare safely with NULL (`a IS b` is never NULL). `IS DISTINCT FROM` is the same idea.
- `NOT IN (subquery)` returns nothing if the subquery produces a NULL. Use `NOT EXISTS`.
- `BETWEEN a AND b` includes both ends.
- `CASE` makes if/else logic inside a query. `COALESCE` supplies defaults. `NULLIF` turns a value into NULL.
- `/` on two integers is integer division. Make one side a real to get decimals.
### Gotchas (the part worth revising twice)
- `x = NULL` and `x != NULL` are never true.
- `NOT (x = 1)` does not return rows where x is NULL.
- `AND` binds tighter than `OR`. Use parentheses.
- `LIKE` is case-insensitive for ASCII in SQLite; `GLOB` is case-sensitive. `%` and `_` inside data need `ESCAPE`.
- Dates stored as text only compare correctly in ISO format `YYYY-MM-DD`. `BETWEEN '2024-01-01' AND '2024-01-31'` misses times later on Jan 31.
- `5 / 2` is 2, not 2.5. `7 % -3` follows the sign of the left side.
- A number and a text value compare by type rank, not by content.
### When it matters
Any WHERE clause touching nullable columns, any report with percentages, any search box.
### One-line recap example
`SELECT first_name, COALESCE(dept, 'Unassigned') FROM employees WHERE (salary BETWEEN 60000 AND 80000) AND first_name LIKE 'a%';`

## 01. Seed data
- A small employee table with some deliberate NULLs (manager, dept) is used throughout.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE employees (
    emp_id     INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name  TEXT NOT NULL,
    dept       TEXT,
    salary     REAL,
    manager_id INTEGER,
    hired      TEXT
);
INSERT INTO employees VALUES
    (1, 'Alice', 'Nguyen', 'Engineering', 95000, NULL, '2019-03-15'),
    (2, 'Bob',   'Smith',  'Engineering', 72000, 1,    '2021-07-01'),
    (3, 'Carla', 'Diaz',   'Sales',       68000, NULL, '2020-11-30'),
    (4, 'Dev',   'Roy',    'Sales',       61000, 3,    '2022-01-10'),
    (5, 'Eve',   'Owusu',  'Marketing',   58000, NULL, '2023-05-20'),
    (6, 'Finn',  'Walsh',  NULL,          75000, 1,    '2024-01-31');
SELECT * FROM employees ORDER BY emp_id;

emp_id | first_name | last_name | dept        | salary  | manager_id | hired     
-------+------------+-----------+-------------+---------+------------+-----------
1      | Alice      | Nguyen    | Engineering | 95000.0 | NULL       | 2019-03-15
2      | Bob        | Smith     | Engineering | 72000.0 | 1          | 2021-07-01
3      | Carla      | Diaz      | Sales       | 68000.0 | NULL       | 2020-11-30
4      | Dev        | Roy       | Sales       | 61000.0 | 3          | 2022-01-10
5      | Eve        | Owusu     | Marketing   | 58000.0 | NULL       | 2023-05-20
6      | Finn       | Walsh     | NULL        | 75000.0 | 1          | 2024-01-31
(6 rows)



## 02. Comparison operators
- `=`, `==`, `!=`, `<>`, `<`, `>`, `<=`, `>=`.
- SQLite has no boolean type: a comparison returns 1 (true), 0 (false) or NULL (unknown).
- Prefer `=` and `<>`: they are standard SQL. SQLite also accepts `==` and `!=`, but other databases may not.

In [3]:
%%sql
SELECT first_name, salary, salary >= 70000 AS is_high FROM employees ORDER BY emp_id;

first_name | salary  | is_high
-----------+---------+--------
Alice      | 95000.0 | 1      
Bob        | 72000.0 | 1      
Carla      | 68000.0 | 0      
Dev        | 61000.0 | 0      
Eve        | 58000.0 | 0      
Finn       | 75000.0 | 1      
(6 rows)



In [4]:
%%sql
SELECT first_name FROM employees WHERE salary >= 70000 AND dept <> 'Sales' ORDER BY emp_id;

first_name
----------
Alice     
Bob       
(2 rows)



## 03. NULL: "unknown", not "empty"
- NULL means "value unknown or missing". It is not 0 and not `''`.
- Any comparison with NULL gives NULL (unknown), including `NULL = NULL`.
- So `WHERE dept = NULL` returns nothing. Use `IS NULL`.
- Arithmetic and `||` with NULL also give NULL: `5 + NULL` is NULL.

In [5]:
%%sql
SELECT NULL = NULL AS eq, NULL <> NULL AS ne, NULL IS NULL AS is_null, 5 + NULL AS plus, 'a' || NULL AS concat;

eq   | ne   | is_null | plus | concat
-----+------+---------+------+-------
NULL | NULL | 1       | NULL | NULL  
(1 row)



In [6]:
%%sql
-- bad: returns nothing
SELECT first_name FROM employees WHERE dept = NULL ORDER BY emp_id;

first_name
----------
(0 rows)



In [7]:
%%sql
-- good
SELECT first_name FROM employees WHERE dept IS NULL ORDER BY emp_id;

first_name
----------
Finn      
(1 row)



In [8]:
%%sql
SELECT first_name, manager_id FROM employees WHERE manager_id IS NOT NULL ORDER BY emp_id;

first_name | manager_id
-----------+-----------
Bob        | 1         
Dev        | 3         
Finn       | 1         
(3 rows)



## 04. Three-valued logic: truth tables
- Values: TRUE (1), FALSE (0), UNKNOWN (NULL).
- `AND`: FALSE wins over UNKNOWN. `OR`: TRUE wins over UNKNOWN. `NOT UNKNOWN` is UNKNOWN.
- `WHERE` keeps a row only when the condition is TRUE, so UNKNOWN rows are dropped, just like FALSE.

In [9]:
%%sql
WITH v(a) AS (VALUES (1), (0), (NULL)),
     w(b) AS (VALUES (1), (0), (NULL))
SELECT a AS A, b AS B, a AND b AS "A AND B", a OR b AS "A OR B"
FROM v, w
ORDER BY a IS NULL, a DESC, b IS NULL, b DESC;

A    | B    | A AND B | A OR B
-----+------+---------+-------
1    | 1    | 1       | 1     
1    | 0    | 0       | 1     
1    | NULL | NULL    | 1     
0    | 1    | 0       | 1     
0    | 0    | 0       | 0     
0    | NULL | 0       | NULL  
NULL | 1    | NULL    | 1     
NULL | 0    | 0       | NULL  
NULL | NULL | NULL    | NULL  
(9 rows)



In [10]:
%%sql
SELECT 1 AS a, NOT 1 AS not_a UNION ALL SELECT 0, NOT 0 UNION ALL SELECT NULL, NOT NULL;

a    | not_a
-----+------
1    | 0    
0    | 1    
NULL | NULL 
(3 rows)



Why `NOT (x = 1)` is not the opposite of `x = 1`:

In [11]:
%%sql
CREATE TABLE flags (id INTEGER PRIMARY KEY, x INTEGER);
INSERT INTO flags (x) VALUES (1), (2), (NULL);
SELECT id, x FROM flags WHERE x = 1 ORDER BY id;

id | x
---+--
1  | 1
(1 row)



In [12]:
%%sql
-- the NULL row is in neither result (`x = 1` gives 1 row, `NOT (x = 1)` gives 1 row): 2 rows in total, not 3
SELECT id, x FROM flags WHERE NOT (x = 1) ORDER BY id;

id | x
---+--
2  | 2
(1 row)



In [13]:
%%sql
-- fix: say what you mean about NULL
SELECT id, x FROM flags WHERE x IS NOT 1 ORDER BY id;

id | x   
---+-----
2  | 2   
3  | NULL
(2 rows)



## 05. IS, IS NOT and IS DISTINCT FROM: NULL-safe comparison
- In SQLite, `a IS b` works like `=` but treats two NULLs as equal and never returns NULL.
- `a IS NOT b` is the NULL-safe `<>`.
- `a IS DISTINCT FROM b` (3.39+) means the same as `IS NOT`. `IS NOT DISTINCT FROM` means the same as `IS`.
- Handy for "has this value changed?" checks and joins on nullable keys.

In [14]:
%%sql
WITH t(a, b) AS (VALUES (1, 1), (1, 2), (1, NULL), (NULL, NULL))
SELECT a, b, a = b AS eq, a IS b AS is_, a IS DISTINCT FROM b AS distinct_from
FROM t
ORDER BY a IS NULL, a, b IS NULL, b;

a    | b    | eq   | is_ | distinct_from
-----+------+------+-----+--------------
1    | 1    | 1    | 1   | 0            
1    | 2    | 0    | 0   | 1            
1    | NULL | NULL | 0   | 1            
NULL | NULL | NULL | 1   | 0            
(4 rows)



In [15]:
%%sql
-- find rows where manager_id differs from a parameter, NULL-safely (param = 1)
SELECT first_name, manager_id FROM employees WHERE manager_id IS DISTINCT FROM 1 ORDER BY emp_id;

first_name | manager_id
-----------+-----------
Alice      | NULL      
Carla      | NULL      
Dev        | 3         
Eve        | NULL      
(4 rows)



## 06. AND, OR, NOT and operator precedence
- Order from tightest to loosest: unary `-`, `||`, `*` `/` `%`, `+` `-`, comparisons (`<`, `=`, `IS`, `IN`, `LIKE`, ...), `NOT`, `AND`, `OR`.
- Careful: in SQLite `||` binds tighter than `*` and `+`. So `1 + 2 || 3` is `1 + (2 || 3)`.
- Mixing AND with OR without parentheses is the classic bug.

In [16]:
%%sql
-- reads as (Engineering AND salary > 80000) OR Sales
SELECT first_name, dept, salary FROM employees
WHERE dept = 'Engineering' AND salary > 80000 OR dept = 'Sales'
ORDER BY emp_id;

first_name | dept        | salary 
-----------+-------------+--------
Alice      | Engineering | 95000.0
Carla      | Sales       | 68000.0
Dev        | Sales       | 61000.0
(3 rows)



In [17]:
%%sql
-- what was probably intended
SELECT first_name, dept, salary FROM employees
WHERE (dept = 'Engineering' OR dept = 'Sales') AND salary > 65000
ORDER BY emp_id;

first_name | dept        | salary 
-----------+-------------+--------
Alice      | Engineering | 95000.0
Bob        | Engineering | 72000.0
Carla      | Sales       | 68000.0
(3 rows)



In [18]:
%%sql
SELECT 2 + 3 * 4 AS mult_first, (2 + 3) * 4 AS parens, 1 + 2 || 3 AS concat_first, NOT 1 = 2 AS not_after_cmp;

mult_first | parens | concat_first | not_after_cmp
-----------+--------+--------------+--------------
14         | 20     | 24           | 1            
(1 row)



## 07. LIKE: patterns, case and escaping
- `%` matches any run of characters (including none). `_` matches exactly one character.
- In SQLite `LIKE` is case-insensitive for ASCII letters by default (not for other alphabets). Other databases may differ.
- To match a literal `%` or `_`, pick an escape character: `LIKE '50!%' ESCAPE '!'`.
- A pattern starting with `%` cannot use an ordinary index (full scan). See 07-Performance/01-Indexes-And-Query-Plans.
- `x LIKE ...` and `x NOT LIKE ...` are both NULL when `x` is NULL, so NULL rows disappear from both.

In [19]:
%%sql
SELECT first_name FROM employees WHERE first_name LIKE 'a%' ORDER BY emp_id;       -- matches Alice despite lowercase a

first_name
----------
Alice     
(1 row)



In [20]:
%%sql
SELECT first_name FROM employees WHERE first_name LIKE '_ob' OR first_name LIKE '%e' ORDER BY emp_id;

first_name
----------
Alice     
Bob       
Eve       
(3 rows)



In [21]:
%%sql
CREATE TABLE promos (id INTEGER PRIMARY KEY, label TEXT);
INSERT INTO promos (label) VALUES ('50% off'), ('50 items'), ('snake_case'), ('snakeXcase');
-- bad: % and _ are wildcards here
SELECT label FROM promos WHERE label LIKE '50% off' OR label LIKE 'snake_case' ORDER BY id;

label     
----------
50% off   
snake_case
snakeXcase
(3 rows)



In [22]:
%%sql
-- good: escape them
SELECT label FROM promos WHERE label LIKE '50!% off' ESCAPE '!' OR label LIKE 'snake!_case' ESCAPE '!' ORDER BY id;

label     
----------
50% off   
snake_case
(2 rows)



In [23]:
%%sql
-- make LIKE case-sensitive when needed (this pragma is deprecated and changes every LIKE on the connection; GLOB is the safer choice)
PRAGMA case_sensitive_like = ON;
SELECT first_name FROM employees WHERE first_name LIKE 'a%' ORDER BY emp_id;

first_name
----------
(0 rows)



In [24]:
%%sql
PRAGMA case_sensitive_like = OFF;
SELECT 'ok' AS reset;

reset
-----
ok   
(1 row)



## 08. GLOB: Unix-style, case-sensitive
- `*` any run, `?` one character, `[abc]` a set, `[a-z]` a range, `[^0-9]` not in set.
- Always case-sensitive. No `ESCAPE` clause: to match a literal `*` put it in brackets: `[*]`.
- Handy for character classes, which LIKE cannot do.

In [25]:
%%sql
SELECT first_name FROM employees WHERE first_name GLOB 'A*' ORDER BY emp_id;        -- only capital A

first_name
----------
Alice     
(1 row)



In [26]:
%%sql
SELECT first_name FROM employees WHERE first_name GLOB '[A-C]*' ORDER BY emp_id;

first_name
----------
Alice     
Bob       
Carla     
(3 rows)



In [27]:
%%sql
WITH t(code) AS (VALUES ('AB-123'), ('ab-123'), ('AB-12'), ('AB-1234'))
SELECT code, code GLOB '[A-Z][A-Z]-[0-9][0-9][0-9]' AS valid_format
FROM t ORDER BY code;

code    | valid_format
--------+-------------
AB-12   | 0           
AB-123  | 1           
AB-1234 | 0           
ab-123  | 0           
(4 rows)



## 09. IN and NOT IN with a list
- `x IN (a, b, c)` is shorthand for `x = a OR x = b OR x = c`.
- `NOT IN` is the negation. Both follow NULL logic: if `x` is NULL the result is NULL.
- IN lists are cleaner than chains of OR.

In [28]:
%%sql
SELECT first_name, dept FROM employees WHERE dept IN ('Sales', 'Marketing') ORDER BY emp_id;

first_name | dept     
-----------+----------
Carla      | Sales    
Dev        | Sales    
Eve        | Marketing
(3 rows)



In [29]:
%%sql
-- Finn (NULL dept) is missing from both IN and NOT IN: NULL is unknown
SELECT first_name, dept FROM employees WHERE dept NOT IN ('Sales', 'Marketing') ORDER BY emp_id;

first_name | dept       
-----------+------------
Alice      | Engineering
Bob        | Engineering
(2 rows)



## 10. IN with a subquery, and the NOT IN + NULL trap
- `x IN (SELECT ...)` filters by a list computed from another query.
- Trap: `x NOT IN (list containing NULL)` can never be TRUE. It is FALSE when x matches, UNKNOWN otherwise. So the whole query returns zero rows.
- Real example: "employees who are not anyone's manager", where `manager_id` has NULLs.
- Fixes: filter the NULLs out in the subquery, or use `NOT EXISTS`. See 02-Querying/03-Subqueries-And-CTEs.

In [30]:
%%sql
-- people who are managers
SELECT first_name FROM employees WHERE emp_id IN (SELECT manager_id FROM employees) ORDER BY emp_id;

first_name
----------
Alice     
Carla     
(2 rows)



In [31]:
%%sql
-- bad: expects Bob, Dev, Eve, Finn... gets nothing, because manager_id contains NULL
SELECT first_name FROM employees WHERE emp_id NOT IN (SELECT manager_id FROM employees) ORDER BY emp_id;

first_name
----------
(0 rows)



In [32]:
%%sql
-- good 1: remove NULLs
SELECT first_name FROM employees
WHERE emp_id NOT IN (SELECT manager_id FROM employees WHERE manager_id IS NOT NULL) ORDER BY emp_id;

first_name
----------
Bob       
Dev       
Eve       
Finn      
(4 rows)



In [33]:
%%sql
-- good 2: NOT EXISTS is NULL-proof
SELECT e.first_name FROM employees e
WHERE NOT EXISTS (SELECT 1 FROM employees m WHERE m.manager_id = e.emp_id) ORDER BY e.emp_id;

first_name
----------
Bob       
Dev       
Eve       
Finn      
(4 rows)



## 11. BETWEEN: inclusive, and on text and dates
- `x BETWEEN a AND b` means `x >= a AND x <= b`. Both ends are included.
- `NOT BETWEEN` is the opposite. Put the smaller value first, or you get no rows.
- Works on text (alphabetical) and ISO dates stored as text.
- Date-time gotcha: `BETWEEN '2024-01-01' AND '2024-01-31'` misses `'2024-01-31 14:00'` since that string is greater than `'2024-01-31'`. Use a half-open range instead: `>= start AND < next_start`.

In [34]:
%%sql
SELECT first_name, salary FROM employees WHERE salary BETWEEN 61000 AND 72000 ORDER BY emp_id;   -- 61000 and 72000 both included

first_name | salary 
-----------+--------
Bob        | 72000.0
Carla      | 68000.0
Dev        | 61000.0
(3 rows)



In [35]:
%%sql
SELECT first_name, last_name FROM employees WHERE last_name BETWEEN 'D' AND 'O' ORDER BY last_name;   -- text range: Diaz..Nguyen (Owusu > 'O')

first_name | last_name
-----------+----------
Carla      | Diaz     
Alice      | Nguyen   
(2 rows)



In [36]:
%%sql
SELECT first_name, hired FROM employees WHERE hired BETWEEN '2020-01-01' AND '2021-12-31' ORDER BY hired;

first_name | hired     
-----------+-----------
Carla      | 2020-11-30
Bob        | 2021-07-01
(2 rows)



In [37]:
%%sql
-- bad vs good with timestamps
CREATE TABLE events (id INTEGER PRIMARY KEY, at TEXT);
INSERT INTO events (at) VALUES ('2024-01-30 09:00'), ('2024-01-31 14:00'), ('2024-02-01 00:00');
SELECT id, at FROM events WHERE at BETWEEN '2024-01-01' AND '2024-01-31' ORDER BY id;        -- misses id 2

id | at              
---+-----------------
1  | 2024-01-30 09:00
(1 row)



In [38]:
%%sql
SELECT id, at FROM events WHERE at >= '2024-01-01' AND at < '2024-02-01' ORDER BY id;       -- half-open: correct

id | at              
---+-----------------
1  | 2024-01-30 09:00
2  | 2024-01-31 14:00
(2 rows)



In [39]:
%%sql
SELECT 5 BETWEEN 10 AND 1 AS reversed_bounds;      -- 0: smaller bound must come first

reversed_bounds
---------------
0              
(1 row)



## 12. CASE: simple and searched forms
- Searched: `CASE WHEN cond THEN v ... ELSE v END`. Conditions are checked top to bottom; first TRUE wins.
- Simple: `CASE expr WHEN value THEN v ... END` compares one expression to values with `=`.
- No `ELSE` and no match gives NULL.
- A simple CASE cannot match NULL (`WHEN NULL` is never true). Use the searched form with `IS NULL`.
- Works in SELECT, WHERE, ORDER BY, and inside aggregates (conditional counts).

In [40]:
%%sql
SELECT first_name, salary,
       CASE WHEN salary >= 90000 THEN 'senior'
            WHEN salary >= 65000 THEN 'mid'
            ELSE 'junior' END AS band
FROM employees ORDER BY emp_id;

first_name | salary  | band  
-----------+---------+-------
Alice      | 95000.0 | senior
Bob        | 72000.0 | mid   
Carla      | 68000.0 | mid   
Dev        | 61000.0 | junior
Eve        | 58000.0 | junior
Finn       | 75000.0 | mid   
(6 rows)



In [41]:
%%sql
SELECT first_name, dept,
       CASE dept WHEN 'Engineering' THEN 'ENG'
                 WHEN 'Sales'       THEN 'SLS'
                 WHEN NULL          THEN 'never matches'
                 ELSE 'other' END AS code,
       CASE WHEN dept IS NULL THEN 'unassigned' ELSE dept END AS dept_label
FROM employees ORDER BY emp_id;

first_name | dept        | code  | dept_label 
-----------+-------------+-------+------------
Alice      | Engineering | ENG   | Engineering
Bob        | Engineering | ENG   | Engineering
Carla      | Sales       | SLS   | Sales      
Dev        | Sales       | SLS   | Sales      
Eve        | Marketing   | other | Marketing  
Finn       | NULL        | other | unassigned 
(6 rows)



In [42]:
%%sql
-- CASE inside an aggregate: conditional counts in one pass
SELECT count(*) AS total,
       sum(CASE WHEN salary >= 70000 THEN 1 ELSE 0 END) AS high_paid,
       sum(CASE WHEN dept IS NULL THEN 1 ELSE 0 END) AS no_dept
FROM employees;

total | high_paid | no_dept
------+-----------+--------
6     | 3         | 1      
(1 row)



In [43]:
%%sql
-- custom sort order with CASE
SELECT first_name, dept FROM employees
ORDER BY CASE dept WHEN 'Sales' THEN 1 WHEN 'Engineering' THEN 2 ELSE 3 END, emp_id;

first_name | dept       
-----------+------------
Carla      | Sales      
Dev        | Sales      
Alice      | Engineering
Bob        | Engineering
Eve        | Marketing  
Finn       | NULL       
(6 rows)



## 13. COALESCE, NULLIF and IIF
- `COALESCE(a, b, c)` returns the first non-NULL argument. Use it for defaults.
- `NULLIF(a, b)` returns NULL if `a = b`, otherwise `a`. Classic use: avoid divide-by-zero, or turn `''` into NULL.
- `IIF(cond, then, else)` (3.32+) is a short searched CASE. `ifnull(a, b)` is a two-argument COALESCE.
- In SQLite, division by zero returns NULL (no error), but in most databases it is an error: `NULLIF` makes the intent explicit.

In [44]:
%%sql
SELECT first_name, COALESCE(dept, 'Unassigned') AS dept, COALESCE(manager_id, 0) AS manager FROM employees ORDER BY emp_id;

first_name | dept        | manager
-----------+-------------+--------
Alice      | Engineering | 0      
Bob        | Engineering | 1      
Carla      | Sales       | 0      
Dev        | Sales       | 3      
Eve        | Marketing   | 0      
Finn       | Unassigned  | 1      
(6 rows)



In [45]:
%%sql
SELECT COALESCE(NULL, NULL, 'third') AS first_non_null, ifnull(NULL, 'fallback') AS two_arg;

first_non_null | two_arg 
---------------+---------
third          | fallback
(1 row)



In [46]:
%%sql
CREATE TABLE stats (name TEXT, hits INTEGER, visitors INTEGER, nickname TEXT);
INSERT INTO stats VALUES ('home', 100, 25, 'start'), ('about', 0, 0, ''), ('blog', 30, 0, NULL);
SELECT name,
       hits * 1.0 / NULLIF(visitors, 0) AS hits_per_visitor,
       COALESCE(NULLIF(nickname, ''), name) AS display_name,
       IIF(hits > 50, 'busy', 'quiet') AS traffic
FROM stats ORDER BY name;

name  | hits_per_visitor | display_name | traffic
------+------------------+--------------+--------
about | NULL             | about        | quiet  
blog  | NULL             | blog         | quiet  
home  | 4.0              | start        | busy   
(3 rows)



## 14. String concatenation with ||
- `a || b` joins text. Numbers are converted to text.
- Any NULL operand makes the whole result NULL. Wrap nullable parts in `COALESCE`.
- `concat()` (3.44+) and `concat_ws(sep, ...)` skip NULLs instead.

In [47]:
%%sql
SELECT first_name || ' ' || last_name AS full_name, first_name || ' / ' || dept AS with_dept
FROM employees ORDER BY emp_id;   -- Finn's with_dept is NULL

full_name    | with_dept          
-------------+--------------------
Alice Nguyen | Alice / Engineering
Bob Smith    | Bob / Engineering  
Carla Diaz   | Carla / Sales      
Dev Roy      | Dev / Sales        
Eve Owusu    | Eve / Marketing    
Finn Walsh   | NULL               
(6 rows)



In [48]:
%%sql
SELECT first_name || ' / ' || COALESCE(dept, '-') AS safe,
       concat_ws(' / ', first_name, dept) AS ws_skips_null
FROM employees ORDER BY emp_id;

safe                | ws_skips_null      
--------------------+--------------------
Alice / Engineering | Alice / Engineering
Bob / Engineering   | Bob / Engineering  
Carla / Sales       | Carla / Sales      
Dev / Sales         | Dev / Sales        
Eve / Marketing     | Eve / Marketing    
Finn / -            | Finn               
(6 rows)



## 15. Integer division vs real, and modulo
- Two integers divided give an integer (truncated toward zero): `5 / 2 = 2`, `-5 / 2 = -2`.
- Make either side real to get decimals: `5 / 2.0`, `5 * 1.0 / 2`, or `CAST(x AS REAL)`.
- `%` is the remainder. Its sign follows the left operand: `-7 % 3 = -1`, `7 % -3 = 1`. It works on integers (reals are cast first).
- Typical uses: even/odd (`n % 2`), every Nth row, wrapping ids into buckets.
- Percentages: multiply by `100.0` first, otherwise integer math wipes out the fraction.

In [49]:
%%sql
SELECT 5 / 2 AS int_div, 5 / 2.0 AS real_div, -5 / 2 AS neg_int_div, CAST(5 AS REAL) / 2 AS cast_div,
       7 % 3 AS mod_pos, -7 % 3 AS mod_neg_left, 7 % -3 AS mod_neg_right, 7.5 % 2 AS mod_real;

int_div | real_div | neg_int_div | cast_div | mod_pos | mod_neg_left | mod_neg_right | mod_real
--------+----------+-------------+----------+---------+--------------+---------------+---------
2       | 2.5      | -2          | 2.5      | 1       | -1           | 1             | 1.0     
(1 row)



In [50]:
%%sql
-- bad vs good percentage: 3 of 8 = 37.5%
SELECT 3 / 8 * 100 AS bad, 3 * 100 / 8 AS int_ok, 3 * 100.0 / 8 AS good;

bad | int_ok | good
----+--------+-----
0   | 37     | 37.5
(1 row)



In [51]:
%%sql
SELECT emp_id, first_name, emp_id % 2 = 0 AS even_id FROM employees ORDER BY emp_id;

emp_id | first_name | even_id
-------+------------+--------
1      | Alice      | 0      
2      | Bob        | 1      
3      | Carla      | 0      
4      | Dev        | 1      
5      | Eve        | 0      
6      | Finn       | 1      
(6 rows)



## 16. Comparing mixed types
- SQLite orders storage classes: NULL < numbers (INTEGER/REAL) < TEXT < BLOB.
- So `'abc' > 999999` is true, and `1 < '1'` is true (a number is always less than text).
- Affinity can soften this: comparing a column with INTEGER/REAL/NUMERIC affinity to a text literal first tries to convert the text to a number. A plain expression like `1 = '1'` has no column and no conversion, so it is false.
- Lesson: store numbers as numbers, and compare like with like. `CAST` explicitly when in doubt.

In [52]:
%%sql
SELECT 1 = '1' AS literal_eq, 1 < '1' AS num_before_text, 'abc' > 999999 AS text_gt_num, x'00' > 'zzz' AS blob_gt_text,
       '10' < '9' AS text_compare, 10 < 9 AS num_compare, CAST('10' AS INTEGER) < CAST('9' AS INTEGER) AS cast_compare;

literal_eq | num_before_text | text_gt_num | blob_gt_text | text_compare | num_compare | cast_compare
-----------+-----------------+-------------+--------------+--------------+-------------+-------------
0          | 1               | 1           | 1            | 1            | 0           | 0           
(1 row)



In [53]:
%%sql
CREATE TABLE typed (n INTEGER, t TEXT);
INSERT INTO typed VALUES (10, '10');
-- column affinity converts the literal: n = '10' is true; t = 10 is also true because TEXT affinity converts 10 to '10'; n = t is true because numeric affinity is applied to the TEXT side
SELECT n = '10' AS int_col_vs_text, t = 10 AS text_col_vs_int, n = t AS col_vs_col FROM typed;

int_col_vs_text | text_col_vs_int | col_vs_col
----------------+-----------------+-----------
1               | 1               | 1         
(1 row)



## 17. Putting it together: a realistic filter
- Combine the pieces and check each NULL branch on purpose.
- Goal: Engineering or unassigned staff, hired 2019-2022, paid 70000+ or with no manager, whose name starts with a letter A-F.

In [54]:
%%sql
SELECT first_name, COALESCE(dept, 'Unassigned') AS dept, salary, manager_id, hired
FROM employees
WHERE (dept = 'Engineering' OR dept IS NULL)
  AND hired >= '2019-01-01' AND hired < '2023-01-01'
  AND (salary >= 70000 OR manager_id IS NULL)
  AND first_name GLOB '[A-F]*'
ORDER BY emp_id;

first_name | dept        | salary  | manager_id | hired     
-----------+-------------+---------+------------+-----------
Alice      | Engineering | 95000.0 | NULL       | 2019-03-15
Bob        | Engineering | 72000.0 | 1          | 2021-07-01
(2 rows)

